# **AdaBoost From Scratch** 
AdaBoost (Adaptive Boosting) is an ensemble technique that:
- Combines multiple weak learners (usually decision stumps)
- Focuses more on misclassified points
- Assigns weights to models (alpha) based on performance
- Final prediction = weighted sum of all weak learners

### **Step 1: Create Dataset**

In [1]:
import pandas as pd
import numpy as np
from mlxtend.plotting import plot_decision_regions

df = pd.DataFrame()

df['X1'] = [1,2,3,4,5,6,6,7,9,9]
df['X2'] = [5,3,6,8,1,9,5,8,9,2]
df['label'] = [1,1,0,1,0,1,0,1,0,0]

df

,X1,X2,label
0,1,5,1
1,2,3,1
2,3,6,0
3,4,8,1
4,5,1,0
5,6,9,1
6,6,5,0
7,7,8,1
8,9,9,0
9,9,2,0


In [2]:
# Convert Labels
X = df[['X1','X2']].values
y = df['label'].values

# Convert 0 → -1
y = np.where(y == 0, -1, 1)

### **Step 2: Build AdaBoost Class**

In [3]:
from sklearn.tree import DecisionTreeClassifier

class AdaBoostCustom:

    def __init__(self, n_estimators=5):
        self.n_estimators = n_estimators
        self.models = []
        self.alphas = []

    def fit(self, X, y):

        n_samples = X.shape[0]

        # Initialize equal weights
        weights = np.ones(n_samples) / n_samples

        for i in range(self.n_estimators):

            # Step 1: Train stump
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=weights)

            y_pred = stump.predict(X)

            # Step 2: Calculate error
            error = np.sum(weights * (y != y_pred))

            # Avoid division by zero
            error = max(error, 1e-10)

            # Step 3: Compute alpha
            alpha = 0.5 * np.log((1 - error) / error)

            # Step 4: Update weights
            weights = weights * np.exp(-alpha * y * y_pred)

            # Normalize weights
            weights = weights / np.sum(weights)

            # Store model + alpha
            self.models.append(stump)
            self.alphas.append(alpha)

            print(f"Stump {i+1} → Error: {error:.4f}, Alpha: {alpha:.4f}")

    def predict(self, X):

        final_pred = np.zeros(X.shape[0])

        for alpha, model in zip(self.alphas, self.models):
            final_pred += alpha * model.predict(X)

        return np.sign(final_pred)

## **Train Model**

In [4]:
model = AdaBoostCustom(n_estimators=5)
model.fit(X, y)

Stump 1 → Error: 0.3000, Alpha: 0.4236
Stump 2 → Error: 0.2143, Alpha: 0.6496
Stump 3 → Error: 0.1364, Alpha: 0.9229
Stump 4 → Error: 0.1842, Alpha: 0.7440
Stump 5 → Error: 0.1774, Alpha: 0.7670


## **Prediction**

In [5]:
pred = model.predict(X)

print("Predictions:", pred)

Predictions: [ 1.  1. -1.  1. -1.  1. -1.  1. -1. -1.]
